# Vector stores and semantic search



In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [5]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata
        self.embedding = None


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document
        self.document_embeddings: list[np.ndarray] = []

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
      self.embedding_model = embedding_model
      self.documents = []
      self.document_embeddings = []

    def add_documents(self, documents):
      texts = [doc.text for doc in documents]
      embeddings = self.embedding_model.encode(texts)

      for doc, emb in zip(documents, embeddings):
          doc.embedding = emb
          self.documents.append(doc)
          self.document_embeddings.append(emb)

    def search(self, query: str, top_k: int = 5) -> list:
      if not self.documents:
          return []

      query_embedding = self.embedding_model.encode([query])

      similarities = cosine_similarity(query_embedding, self.document_embeddings)[0]

      top_indices = np.argsort(similarities)[::-1][:top_k]

      results = []
      for idx in top_indices:
          results.append(SearchResult(similarities[idx], self.documents[idx]))

      return results

In [6]:
fun_facts_doc = pd.read_csv('animal-fun-facts-dataset.csv')
fun_facts_doc.head()

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


In [7]:
# Carga de datos
documents = []
for _, row in fun_facts_doc.iterrows():
    if pd.notna(row['text']):
        metadata = {
            'animal_name': row['animal_name'],
            'source': row['source'] if pd.notna(row['source']) else '',
            'media_link': row['media_link'] if pd.notna(row['media_link']) else '',
            'wikipedia_link': row['wikipedia_link'] if pd.notna(row['wikipedia_link']) else ''
        }
        documents.append(Document(row['text'], metadata))

model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = VectorStore(model)
vector_store.add_documents(documents)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Ejemplos

searches = [
  "biggest weight",
  "longest sleep time",
  "animals with long limbs",
  "animals that jump the highest",
  "slowest birthrate of a species"
]

# Busquedas
for search in searches:
  results = vector_store.search(search, top_k=3)
  print(f"SEARCH: {search}")
  for r in results:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()
  print('---')


SEARCH: biggest weight
Score: 0.5675 - great bustard: The great bustard is the heaviest bird in the world capable of flight. It can we...
Metadata:  {'animal_name': 'great bustard', 'source': 'https://reddit.com/r/Awwducational/comments/yfz0q6/the_great_bustard_is_the_heaviest_bird_in_the/', 'media_link': 'https://i.redd.it/k5ooikby8nw91.png', 'wikipedia_link': '/wiki/Great_bustard'}

Score: 0.5498 - grant's gazelle: Adult males are seen as the largest gazelle concerning body weight....
Metadata:  {'animal_name': "grant's gazelle", 'source': 'https://seaworld.org/animals/facts/mammals/grants-gazelle/', 'media_link': '', 'wikipedia_link': '/wiki/Grant%27s_gazelle'}

Score: 0.5331 - sperm whale: They are the largest of the toothed whales.
Adult males can weigh up to 80 tonne...
Metadata:  {'animal_name': 'sperm whale', 'source': 'https://factanimal.com/sperm-whale/', 'media_link': '', 'wikipedia_link': '/wiki/Sperm_whale'}

---
SEARCH: longest sleep time
Score: 0.5063 - koala: A koala sp

## Part II: Filtering by metadata

In [9]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.document_embeddings = []

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        embeddings = self.embedding_model.encode(texts)

        for doc, emb in zip(documents, embeddings):
            doc.embedding = emb
            self.documents.append(doc)
            self.document_embeddings.append(emb)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if not self.documents:
            return []

        # Metadata filters
        if metadata_filter is not None:
            filtered_docs = []
            filtered_embeddings = []

            for idx, doc in enumerate(self.documents):
                match = True
                for key, value in metadata_filter.items():
                    if key not in doc.metadata or doc.metadata[key] != value:
                        match = False
                        break

                if match:
                    filtered_docs.append(doc)
                    filtered_embeddings.append(self.document_embeddings[idx])
        else:
            # Si no hubo filtro aplicado, se usan todos los documentos
            filtered_docs = self.documents
            filtered_embeddings = self.document_embeddings

        if not filtered_docs:
            return []  # No hubo documentos que matchearan el filtro

        query_embedding = self.embedding_model.encode([query])
        similarities = cosine_similarity(query_embedding, filtered_embeddings)[0]

        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append(SearchResult(similarities[idx], filtered_docs[idx]))

        return results

In [10]:
# Cargar documentos
filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(documents)

In [11]:
# Búsquedas

results_1 = filtered_store.search(
    query="teeth",
    top_k=3,
    metadata_filter={"animal_name": "aardvark"}
)


print("SEARCH: 'teeth'\nMETADATA: animal_name = aardvark")
for r in results_1:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()

results_2 = filtered_store.search(
    query="height",
    top_k=3,
    metadata_filter={"animal_name": "elephant"}
)

print('---')

print("SEARCH: 'height'\nMETADATA: animal_name = elephant")
for r in results_2:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()

results_3 = filtered_store.search(
    query="eating",
    top_k=3,
    metadata_filter={"animal_name": "whale"}
)

print('---')

print("SEARCH: 'eating'\nMETADATA: animal_name = whale")
for r in results_3:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()

results_4 = filtered_store.search(
    query="babies",
    top_k=3,
    metadata_filter={"animal_name": "snake"}
)

print('---')

print("SEARCH: 'babies'\nMETADATA: animal_name = snake")
for r in results_4:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()

results_5 = filtered_store.search(
    query="carnivore",
    top_k=3,
    metadata_filter={"animal_name": "tiger"}
)

print('---')

print("SEARCH: 'carnivore'\nMETADATA: animal_name = tiger")
for r in results_5:
    print(f"Score: {r.score:.4f} - {r.document.metadata['animal_name']}: {r.document.text[:80]}...")
    print("Metadata: ", r.document.metadata)
    print()



SEARCH: 'teeth'
METADATA: animal_name = aardvark
Score: 0.4454 - aardvark: Baby aardvarks are born with front teeth that fall out and
never grow back....
Metadata:  {'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Score: 0.4280 - aardvark: Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel....
Metadata:  {'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Score: 0.3197 - aardvark: Aardvarks are also named after their long mouths.
This name refers to the jaw of...
Metadata:  {'animal_name': 'aardvark', 'source': 'https://factanimal.com/aardvark/', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

---
SEARCH: 'height'
METADATA: animal_name = elephant
Score: 0.2449 - elephant: The largest bull African elephants can have 10 foot long tusks and must be